# Introduction

In [1]:
# Safer eval: force CPU, low threads, and per-eqn fresh env to avoid native crashes.
# Run from repo root.

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # avoid hard MPS usage
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd
import torch

from stable_baselines3 import PPO
from envs.env_multi_eqn_fixed import multiEqn

# ----------------------------
# Config (EDIT if needed)
# ----------------------------
NO_BUFFER_ROOT = Path("data/abel_level2_hidden_dim256_nenvs1")  # non-buffer runs
AGENT_NAME     = "ppo-tree"
GEN            = "abel_level2"
MODEL_BASENAMES = ["final_model.zip", "model.zip"]  # preference order
MAX_STEPS      = 60
DETERMINISTIC  = True

# ----------------------------
# Helpers
# ----------------------------
def find_model(root: Path, agent: str) -> Path:
    agent_dir = root / agent
    if not agent_dir.exists():
        raise FileNotFoundError(f"Agent dir not found: {agent_dir.resolve()}")

    candidates = []
    # Prefer seed subdirs
    for sd in sorted(agent_dir.glob("seed*")):
        for bn in MODEL_BASENAMES:
            p = sd / bn
            if p.exists():
                candidates.append(p)
        candidates.extend(sd.glob("seed*.zip"))  # SB3's default export names

    # Fallback: anywhere under agent_dir
    if not candidates:
        for bn in MODEL_BASENAMES:
            candidates.extend(agent_dir.rglob(bn))
        candidates.extend(agent_dir.rglob("seed*.zip"))

    if not candidates:
        raise FileNotFoundError(f"No model files under {agent_dir.resolve()}")

    best = max(candidates, key=lambda p: p.stat().st_mtime)
    print(f"✓ Using model: {best}")
    return best

def make_env(gen: str):
    # Match training settings for ppo-tree (graph obs, curriculum on)
    env = multiEqn(
        gen=gen,
        use_relabel_constants=False,
        state_rep="graph_integer_1d",
        sparse_rewards=False,
        use_curriculum=True,
    )
    try:
        env.reset(seed=0)
    except TypeError:
        env.seed(0)
    return env

def pin_equation(env, eqn):
    # Use the env’s helper if present
    if callable(getattr(env, "set_equation", None)):
        env.set_equation(eqn)
        if callable(getattr(env, "setup", None)):
            env.setup()
        # Many of your envs expose `state` that the policy expects:
        s = getattr(env, "state", None)
        if s is not None:
            return s
    # Fallback: just use the reset obs
    obs, _ = env.reset()
    return obs

@torch.no_grad()
def greedy_solve_one(model: PPO, gen: str, eqn, max_steps=60, deterministic=True):
    # Fresh env for this eqn to avoid any stale internal state
    env = make_env(gen)
    try:
        obs = pin_equation(env, eqn)
        last_info = {}
        for t in range(1, max_steps + 1):
            # Ensure numpy array (SB3 handles dict/np);
            # no gradients and CPU device to avoid MPS
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, _, terminated, truncated, info = env.step(int(action))
            last_info = info
            if info.get("is_solved", False):
                return True, t, last_info
            if terminated or truncated:
                break
        return False, t, last_info
    finally:
        try:
            env.close()
        except Exception:
            pass
        gc.collect()

# ----------------------------
# Run evaluation (CPU only)
# ----------------------------
torch.set_num_threads(1)

model_path = find_model(NO_BUFFER_ROOT, AGENT_NAME)

# Build a temp env to ensure the obs space matches before loading
_tmp_env = make_env(GEN)
# Force CPU to dodge MPS crashes on macOS
model = PPO.load(model_path, env=_tmp_env, device="cpu", print_system_info=False)
# Keep the temporary env only to satisfy SB3’s policy spaces; we’ll create fresh envs per eqn.

test_eqns = getattr(_tmp_env, "test_eqns", None)
if not test_eqns:
    raise RuntimeError("Env has no test_eqns; check generator paths (equation_templates/*).")

rows = []
solved_count = 0

for i, eqn in enumerate(test_eqns, 1):
    print(eqn)
    ok, steps, info = greedy_solve_one(model, GEN, eqn, max_steps=MAX_STEPS, deterministic=DETERMINISTIC)
    solved_count += int(ok)
    lhs = info.get("lhs")
    rhs = info.get("rhs")
    rows.append({
        "idx": i,
        "equation": str(eqn),
        "solved": bool(ok),
        "steps": int(steps),
        "lhs": str(lhs) if lhs is not None else "",
        "rhs": str(rhs) if rhs is not None else "",
    })
    print(f"{'✓' if ok else '✗'} [{i:03d}/{len(test_eqns)}] {eqn}  (steps={steps})")

df = pd.DataFrame(rows).sort_values(["solved", "idx"], ascending=[False, True])
display(df.head(10))

print("\n=== Summary ===")
print(f"Solved {solved_count} / {len(test_eqns)} ({solved_count/len(test_eqns):.2%})")

# Save next to the model checkpoint
out_csv = model_path.parent / "test_eval_detail.csv"
df.to_csv(out_csv, index=False)
print(f"Saved per-equation results → {out_csv}")

unsolved = df[df["solved"] == False]["equation"].tolist()
if unsolved:
    print("\nUnsolved equations:")
    for u in unsolved[:50]:
        print(" -", u)
    if len(unsolved) > 50:
        print(f"... and {len(unsolved)-50} more")